# 面向对象里面使用到的装饰器

## 属性访问
- `@property`
- `@my_property.setter`
- `@my_property.deleter`

- @property
> 1. 让这个方法变成属性，可供外部访问（从 function 变成 parameter）
> 2. 也可以吧`@property` 理解成一个 getter 方法
> 3. 对应这个 getter 方法，还有`@my_property.setter`注解
> 4. 这样子就有了 getter 和 setter 两个方法了
> 5. 除此之外还有`@my_property.deleter`

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height
    # 这里矩形类本身不具有面积属性，只有长和宽
    # 要求面积，需要对原有的长宽这两个属性做逻辑处理（就是乘法）
    # 于是得到一个新的属性
    @property
    def area(self):
        return self.width * self.height

my_rectangle = Rectangle(2,3)
print(my_rectangle.area)

In [21]:
class animal:
    def __init__(self, _name):
        self._name = _name
    @property
    def name(self):
        return self._name
    @name.setter
    def name(self, new_name):
        if new_name != self._name:
            print("You changed its name")
        self._name = new_name
    @name.deleter
    def name(self):
        print("You deleted its name")
        del self._name

dog = animal("Dog")
print(dog.name)
print(dog._name)

dog.name = "Cat"
print(dog.name)

dog._name = "Dog"
print(dog._name)

del dog.name
try:
    print(dog.name)
except AttributeError as e:
    print(f"Exception Occurred: {e}")

Dog
Dog
You changed its name
Cat
Dog
You deleted its name
Exception Occurred: 'animal' object has no attribute '_name'


## 方法
- `@staticmethod`静态方法
- `@classmethod`类方法
- `@abstractmethod`抽象方法（强制子类实现）

## 强制生成样板代码 `@dataclass`
> 1. 自动生成 `__init__`、`__repr__`、`__eq__` 等方法
> 2. 只看带类型注解的作为字段（只有这些字段会纳入自动生成的样板里面）

In [31]:
from dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int = 0

my_point = Point(2)
your_point = Point(2)
print("(",my_point.x, ",", my_point.y,")")
print(my_point.__repr__())
print(my_point == your_point)


( 2 , 0 )
Point(x=2, y=0)
True


In [24]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

    def __eq__(self, other):
        return isinstance(other, Point) and (self.x, self.y) == (other.x, other.y)
my_point = Point(1,2)
your_point = Point(1,2)
print(my_point.x, "\t", my_point.y)
print(my_point.__repr__())
print(my_point == your_point)

1 	 2
Point(x=1, y=2)
True


### __`@dataclass` 里面的各种声明__

In [32]:
from dataclasses import dataclass, field
from typing import ClassVar

@dataclass
class C:
    a: int                      # 字段：无默认值
    b: int = 0                  # 字段：有默认值
    c = 0                       # 不是字段：普通类属性
    d: ClassVar[int] = 0        # 不是字段：类变量
    e: list = field(default_factory=list)  # 字段：默认工厂

In [34]:
try:
    print(C.a)
except AttributeError as e:
    print(f"Exception Occurred: {e}")

try:
    print(C.b)
except AttributeError as e:
    print(f"Exception Occurred: {e}")

try:
    print(C.c)
except AttributeError as e:
    print(f"Exception Occurred: {e}")

try:
    print(C.d)
except AttributeError as e:
    print(f"Exception Occurred: {e}")

try:
    print(C.e)
except AttributeError as e:
    print(f"Exception Occurred: {e}")


Exception Occurred: type object 'C' has no attribute 'a'
0
0
0
Exception Occurred: type object 'C' has no attribute 'e'


In [36]:
my_c = C(0,1,2)

try:
    print(my_c.a)
except AttributeError as e:
    print(f"Exception Occurred: {e}")

try:
    print(my_c.b)
except AttributeError as e:
    print(f"Exception Occurred: {e}")

try:
    print(my_c.c)
except AttributeError as e:
    print(f"Exception Occurred: {e}")

try:
    print(my_c.d)
except AttributeError as e:
    print(f"Exception Occurred: {e}")

try:
    print(my_c.e)
except AttributeError as e:
    print(f"Exception Occurred: {e}")


0
1
0
0
2


- 这里展示一下 DS 写的两种等价版本来辅助理解

In [37]:
from __future__ import annotations

from dataclasses import dataclass, field, InitVar
from typing import ClassVar
import hashlib


@dataclass
class User:
    name: str
    password_hash: str = field(init=False, repr=False)
    age: int = 0
    tags: list[str] = field(default_factory=list)
    password: InitVar[str] = ""
    species: ClassVar[str] = "human"

    def __post_init__(self, password: str) -> None:
        if self.age < 0:
            raise ValueError("age must be >= 0")
        self.password_hash = hashlib.sha256(password.encode()).hexdigest()

    @property
    def display_name(self) -> str:
        return f"{self.name} ({self.age})"

In [38]:
from __future__ import annotations

from typing import ClassVar
import hashlib

_MISSING = object()


class User:
    # ClassVar：类变量，不是实例字段
    species: ClassVar[str] = "human"

    def __init__(
        self,
        name: str,
        age: int = 0,
        tags: list[str] | object = _MISSING,
        password: str = "",
    ) -> None:
        self.name = name
        self.age = age

        # 等价于 field(default_factory=list)
        if tags is _MISSING:
            tags = []
        self.tags = tags

        # dataclass 会在 __init__ 末尾自动调用 __post_init__
        # 并把 InitVar 参数传进去
        self.__post_init__(password)

    def __post_init__(self, password: str) -> None:
        if self.age < 0:
            raise ValueError("age must be >= 0")
        self.password_hash = hashlib.sha256(password.encode()).hexdigest()

    @property
    def display_name(self) -> str:
        return f"{self.name} ({self.age})"

    # 等价于 dataclass 自动生成的 __repr__
    # 注意：password_hash 设置了 repr=False，所以不显示
    # password 是 InitVar，不参与 repr
    # species 是 ClassVar，不参与 repr
    def __repr__(self) -> str:
        return f"User(name={self.name!r}, age={self.age!r}, tags={self.tags!r})"

    # 等价于 dataclass 自动生成的 __eq__
    # 参与比较的是所有普通字段：name、password_hash、age、tags
    # InitVar 和 ClassVar 不参与
    def __eq__(self, other: object) -> bool:
        if other.__class__ is self.__class__:
            return (
                self.name,
                self.password_hash,
                self.age,
                self.tags,
            ) == (
                other.name,
                other.password_hash,
                other.age,
                other.tags,
            )
        return NotImplemented

    # dataclass 默认 eq=True、frozen=False 时，__hash__ 被设为 None
    __hash__ = None